### Caso: Embotelladora SierraAzul, S.A. de C.V.

Embotelladora SierraAzul, S.A. de C.V., es una empresa mexicana independiente que produce y distribuye bebidas gaseosas, agua saborizada y energéticas. Su centro corporativo está en Ciudad de México, con dos plantas principales de producción (CDMX y Monterrey) y una red de centros de distribución que cubre buena parte del país. SierraAzul compite contra gigantes globales, por lo que su rentabilidad depende muchísimo de controlar bien sus precios y descuentos comerciales.

La empresa atiende varios canales: autoservicio (cadenas de supermercados y clubes de precio), canal tradicional (tiendas de barrio, misceláneas), mayoristas/distribuidores regionales y cadenas regionales de supermercados. Cada canal tiene condiciones distintas de negociación: porcentajes de descuento, requisitos de volumen, políticas para combos multisabor y lineamientos específicos para lanzamientos de nuevos productos. Todo esto está documentado en una serie de políticas comerciales internas que se actualizan de forma periódica.

El problema es que estas políticas están repartidas en varios documentos y no siempre son fáciles de consultar. El equipo comercial —ejecutivos de venta, key account managers y supervisores de zona— vive haciendo preguntas como:

-   “¿Hasta cuánto descuento puedo ofrecer a un mayorista si me compra un pallet completo?”

-   “¿Puedo aplicar combos multisabor en canal tradicional con este cliente?”

-   “Este cliente es una cadena regional, ¿qué límite de descuento tengo para un lanzamiento?”

Hoy, muchas de esas dudas se resuelven por correo, llamadas o mensajes de WhatsApp al área de Pricing y Trade Marketing. Eso genera retrasos en la negociación, respuestas inconsistentes entre ejecutivos y un riesgo permanente de ofrecer descuentos por encima de lo permitido, afectando el margen de contribución.

Ante esta situación, la Dirección Comercial y el área de Analytics quieren probar un asistente interno que permita a los usuarios escribir preguntas en lenguaje natural (por ejemplo: “¿Qué descuento máximo puedo ofrecer a una cadena regional en un lanzamiento?”) y que el sistema recupere los fragmentos de política más relevantes para ayudar a responder de forma rápida y consistente. La idea no es reemplazar el criterio del ejecutivo, sino darle una herramienta que le muestre las políticas correctas en segundos.

Como primer piloto, han decidido trabajar únicamente con un subconjunto acotado de documentos:

- Políticas de descuentos por canal (autoservicio, tradicional, mayoristas, cadenas regionales).

-   Lineamientos para combos multisabor.

-   Política específica para lanzamientos de nuevos productos.

-   Lineamientos generales de control de descuentos y rentabilidad mínima.

El objetivo de este caso es que, a partir de estos textos, se construya una pequeña base vectorial de conocimiento comercial que permita responder preguntas frecuentes sobre descuentos y políticas, y que sirva como base para un futuro asistente de precios para Embotelladora SierraAzul.

Tarea: Asistente interno de políticas comerciales para Embotelladora SierraAzul

A partir del caso de Embotelladora SierraAzul y de los textos de políticas comerciales que te proporciono en este cuaderno (documentos internos de descuentos, combos, lanzamientos, etc.) deberás construir una pequeña base vectorial que permita responder preguntas frecuentes sobre descuentos y condiciones comerciales.

### Parte A

In [1]:
import os
import chromadb

In [3]:

#* Persistencia de datos
PERSIST_DIR = "./db_chroma_embotelladora_sierrazul"

os.makedirs(PERSIST_DIR, exist_ok=True)

In [4]:
documentos_sierraazul = [
    {
        "id": "descuentos_autoservicio",
        "texto": """
    Política de descuentos para canal autoservicio (supermercados y grandes cadenas).
    Descuento base: 8% sobre precio de lista para productos del portafolio estándar.
    Se permite llegar hasta 12% en negociaciones anuales con compromiso de volumen.
    Los descuentos adicionales deben estar ligados a exhibiciones especiales o puntas de góndola aprobadas por Trade Marketing.
    """,
    },
    {
        "id": "descuentos_tradicional",
        "texto": """
    Política de descuentos para canal tradicional (tiendas de barrio y misceláneas).
    Descuento base: 5% sobre precio de lista para cajas surtidas.
    Se permite hasta 7% si el cliente alcanza el volumen mínimo mensual definido por zona.
    Los descuentos deben registrarse en el sistema como “descuento por volumen canal tradicional”.
    """,
    },
    {
        "id": "descuentos_mayoristas",
        "texto": """
    Política de descuentos para mayoristas y distribuidores regionales.
    Descuento base: 10% sobre precio de lista para compra mínima de pallet completo.
    Se puede llegar hasta 15% para clientes que superen el volumen trimestral pactado.
    Todo descuento superior a 12% requiere aprobación de Finanzas y Dirección Comercial.
    """,
    },
    {
        "id": "descuentos_cadenas_regionales",
        "texto": """
    Política de descuentos para cadenas regionales de autoservicio.
    Descuento base: 9% sobre precio de lista.
    Se permite negociar hasta 14% cuando la cadena garantiza exclusividad de portafolio en ciertas categorías.
    Los acuerdos deben formalizarse en contratos anuales con revisión de rentabilidad por SKU.
    """,
    },
    {
        "id": "combos_multisabor",
        "texto": """
    Lineamientos para combos y promociones multisabor.
    Los combos deben mezclar al menos 3 sabores de la misma familia de producto.
    El descuento sobre el combo no debe superar el 10% del valor total de los productos individuales.
    Los combos sólo aplican en canal tradicional y autoservicio, no en mayoristas.
    """,
    },
    {
        "id": "lanzamientos_nuevos_productos",
        "texto": """
    Política de precios y descuentos para lanzamientos.
    Durante los primeros 3 meses, el precio de lanzamiento debe ser igual o mayor al precio de lista estándar.
    Se permite un descuento promocional máximo del 5% para introducir el producto en cadenas clave.
    Pasado el periodo de lanzamiento, los descuentos deben alinearse con la política del canal correspondiente.
    """,
    },
    {
        "id": "lineamientos_generales_descuentos",
        "texto": """
    Lineamientos generales de rentabilidad y control de descuentos.
    Ningún cliente puede acumular más de 3 tipos de descuentos simultáneos sobre el mismo SKU.
    El margen bruto mínimo después de descuentos debe ser del 18%.
    Finanzas revisa trimestralmente la rentabilidad por cliente y puede bloquear esquemas no sostenibles.
    """,
    },
]

In [5]:
docs_politicas_sierrazul = {
    "ids": [doc['id'] for doc in documentos_sierraazul],
    "documents": [doc['texto'] for doc in documentos_sierraazul]
}

In [6]:

#* Generamos una función para mejorar la respuesta de las consultas
def imprimir_resultados(results, titulo="Resultados"):
    print("="*90)
    print(titulo)
    print("="*90)
    
    ids = results.get("ids", [[]])[0]
    documents = results.get("documents", [[]])[0]
    dists = results.get("distances", [[]])[0]
    
    for i, (rid, rdoc, rdist) in enumerate(zip(ids, documents, dists), start=1):
        print(f"Top {i} | ID: {rid:<28} | Distancia: {rdist:.4f}")
        snippet = (rdoc[:180] + '...') if len(rdoc) > 180 else rdoc
        print(f"         | Documento: {snippet}")
        print("-"*90)
    
    print("Nota: menor distancia indica mayor similitud.")

In [7]:
client_sierrazul = chromadb.PersistentClient(path=PERSIST_DIR)

collection = client_sierrazul.get_or_create_collection(
    name="politicas_embotelladora_sierrazul",
)

print(f"Conexión establecida con la colección '{collection.name}' en el directorio de persistencia '{PERSIST_DIR}'")

Conexión establecida con la colección 'politicas_embotelladora_sierrazul' en el directorio de persistencia './db_chroma_embotelladora_sierrazul'


In [10]:
collection.upsert(
    ids=docs_politicas_sierrazul['ids'],
    documents=docs_politicas_sierrazul['documents'],
    metadatas=docs_politicas_sierrazul.get("metadatas")
)

print(f"Documentos insertados/actualizados en la colección '{collection.name}'")

Documentos insertados/actualizados en la colección 'politicas_embotelladora_sierrazul'


In [11]:
consulta_1 = "¿Qué descuento máximo puede ofrecer a un mayorista en un pedido grande?"
res_rel = collection.query(query_texts=[consulta_1], n_results=3)
imprimir_resultados(res_rel, "Descuentos máximos por pedido grande")

Descuentos máximos por pedido grande
Top 1 | ID: descuentos_mayoristas        | Distancia: 0.8450
         | Documento: 
    Política de descuentos para mayoristas y distribuidores regionales.
    Descuento base: 10% sobre precio de lista para compra mínima de pallet completo.
    Se puede llegar ha...
------------------------------------------------------------------------------------------
Top 2 | ID: lanzamientos_nuevos_productos | Distancia: 0.8518
         | Documento: 
    Política de precios y descuentos para lanzamientos.
    Durante los primeros 3 meses, el precio de lanzamiento debe ser igual o mayor al precio de lista estándar.
    Se permi...
------------------------------------------------------------------------------------------
Top 3 | ID: descuentos_tradicional       | Distancia: 0.9515
         | Documento: 
    Política de descuentos para canal tradicional (tiendas de barrio y misceláneas).
    Descuento base: 5% sobre precio de lista para cajas surtidas.
    Se perm

In [12]:
consulta_2 = "¿Puedo aplicar combos multisabor en el canal tradicional?"
res_rel = collection.query(query_texts=[consulta_2], n_results=3)
imprimir_resultados(res_rel, "Politica multisabor")

Politica multisabor
Top 1 | ID: combos_multisabor            | Distancia: 0.6754
         | Documento: 
    Lineamientos para combos y promociones multisabor.
    Los combos deben mezclar al menos 3 sabores de la misma familia de producto.
    El descuento sobre el combo no debe sup...
------------------------------------------------------------------------------------------
Top 2 | ID: descuentos_tradicional       | Distancia: 0.7851
         | Documento: 
    Política de descuentos para canal tradicional (tiendas de barrio y misceláneas).
    Descuento base: 5% sobre precio de lista para cajas surtidas.
    Se permite hasta 7% si e...
------------------------------------------------------------------------------------------
Top 3 | ID: descuentos_autoservicio      | Distancia: 0.7864
         | Documento: 
    Política de descuentos para canal autoservicio (supermercados y grandes cadenas).
    Descuento base: 8% sobre precio de lista para productos del portafolio estándar.
    Se p.